In [97]:
import torch
import torch.nn as nn
import numpy as np

In [98]:
def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B, C = X.shape[:2]
    #print(f'B: {B}, C: {C}')
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    #print(f'outh: {out_H}, out_W: {out_W}')

    cols = []

    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W * stride, stride):
                patch = X_padded[b, :, i:i+kH, j:j+kW].ravel()
                cols.append(patch)
    
    return np.array(cols).reshape(B, -1, out_H*out_W), out_H, out_W

In [99]:
torch.manual_seed(1)

unfold = nn.Unfold(kernel_size=(2, 2), stride=1, padding=0)
input = torch.tensor([[[[1, 2, 3, 4, 5],
                               [6, 7, 8, 9, 10],
                               [11, 12, 13, 14, 15],
                               [16, 17, 18, 19, 20],
                               [21, 22, 23, 24, 25]],

                              [[26, 27, 28, 29, 30],
                               [31, 32, 33, 34, 35],
                               [36, 37, 38, 39, 40],
                               [41, 42, 43, 44, 45],
                               [46, 47, 48, 49, 50]],

                              [[51, 52, 53, 54, 55],
                               [56, 57, 58, 59, 60],
                               [61, 62, 63, 64, 65],
                               [66, 67, 68, 69, 70],
                               [71, 72, 73, 74, 75]]],

                            [[[76, 77, 78, 79, 80],
                               [81, 82, 83, 84, 85],
                               [86, 87, 88, 89, 90],
                               [91, 92, 93, 94, 95],
                               [96, 97, 98, 99, 100]],

                              [[101, 102, 103, 104, 105],
                               [106, 107, 108, 109, 110],
                               [111, 112, 113, 114, 115],
                               [116, 117, 118, 119, 120],
                               [121, 122, 123, 124, 125]],

                              [[126, 127, 128, 129, 130],
                               [131, 132, 133, 134, 135],
                               [136, 137, 138, 139, 140],
                               [141, 142, 143, 144, 145],
                               [146, 147, 148, 149, 150]]]], dtype=torch.float32)

print(input.shape)
output = unfold(input)
print(output.shape)

torch.Size([2, 3, 5, 5])
torch.Size([2, 12, 16])


In [100]:
new, outh, outw = im2col_multi(input, kernel_shape=(2, 2), stride=1, padding=0)

print(new.shape)

(2, 12, 16)


In [101]:
print(output[0])

tensor([[ 1.,  2.,  3.,  4.,  6.,  7.,  8.,  9., 11., 12., 13., 14., 16., 17.,
         18., 19.],
        [ 2.,  3.,  4.,  5.,  7.,  8.,  9., 10., 12., 13., 14., 15., 17., 18.,
         19., 20.],
        [ 6.,  7.,  8.,  9., 11., 12., 13., 14., 16., 17., 18., 19., 21., 22.,
         23., 24.],
        [ 7.,  8.,  9., 10., 12., 13., 14., 15., 17., 18., 19., 20., 22., 23.,
         24., 25.],
        [26., 27., 28., 29., 31., 32., 33., 34., 36., 37., 38., 39., 41., 42.,
         43., 44.],
        [27., 28., 29., 30., 32., 33., 34., 35., 37., 38., 39., 40., 42., 43.,
         44., 45.],
        [31., 32., 33., 34., 36., 37., 38., 39., 41., 42., 43., 44., 46., 47.,
         48., 49.],
        [32., 33., 34., 35., 37., 38., 39., 40., 42., 43., 44., 45., 47., 48.,
         49., 50.],
        [51., 52., 53., 54., 56., 57., 58., 59., 61., 62., 63., 64., 66., 67.,
         68., 69.],
        [52., 53., 54., 55., 57., 58., 59., 60., 62., 63., 64., 65., 67., 68.,
         69., 70.],
        [5

In [102]:
print(new[0])

[[ 1.  2.  6.  7. 26. 27. 31. 32. 51. 52. 56. 57.  2.  3.  7.  8.]
 [27. 28. 32. 33. 52. 53. 57. 58.  3.  4.  8.  9. 28. 29. 33. 34.]
 [53. 54. 58. 59.  4.  5.  9. 10. 29. 30. 34. 35. 54. 55. 59. 60.]
 [ 6.  7. 11. 12. 31. 32. 36. 37. 56. 57. 61. 62.  7.  8. 12. 13.]
 [32. 33. 37. 38. 57. 58. 62. 63.  8.  9. 13. 14. 33. 34. 38. 39.]
 [58. 59. 63. 64.  9. 10. 14. 15. 34. 35. 39. 40. 59. 60. 64. 65.]
 [11. 12. 16. 17. 36. 37. 41. 42. 61. 62. 66. 67. 12. 13. 17. 18.]
 [37. 38. 42. 43. 62. 63. 67. 68. 13. 14. 18. 19. 38. 39. 43. 44.]
 [63. 64. 68. 69. 14. 15. 19. 20. 39. 40. 44. 45. 64. 65. 69. 70.]
 [16. 17. 21. 22. 41. 42. 46. 47. 66. 67. 71. 72. 17. 18. 22. 23.]
 [42. 43. 47. 48. 67. 68. 72. 73. 18. 19. 23. 24. 43. 44. 48. 49.]
 [68. 69. 73. 74. 19. 20. 24. 25. 44. 45. 49. 50. 69. 70. 74. 75.]]
